# Train Qud-1

Fine-tunes Qud-1 (LoRA on Qwen2.5-1.5B-Instruct), publishes the weights to Hugging Face and builds GGUF files for the free Space that serves it.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (Colab), or *Accelerator → GPU T4* (Kaggle).
2. Add secrets (Colab: the 🔑 panel; Kaggle: *Add-ons → Secrets*):
   - `HF_TOKEN`: a Hugging Face token with **write** access (huggingface.co/settings/tokens).
   - `GROQ_API_KEY`: optional, a free key from console.groq.com, used to distil extra training data.
3. *Run all.* It takes about 30–60 minutes.

In [ ]:
# Get the training kit
!git clone -q --depth 1 https://github.com/Quddos/quddus
%cd quddus
!test -d training/qud-1 || (git fetch -q --depth 1 origin claude/lucid-brown-cv1gsn && git checkout -q FETCH_HEAD)
%cd training/qud-1
!pip install -q -r requirements.txt

In [ ]:
# Load secrets from Colab or Kaggle
import os
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
os.environ["TEACHER_API_KEY"] = secret("GROQ_API_KEY") or ""
print("HF token:", "set" if os.environ["HF_TOKEN"] else "MISSING (weights won't be published)")
print("Teacher key:", "set" if os.environ["TEACHER_API_KEY"] else "not set (training on the hand-written set only)")

## 1. Distil extra examples (optional)

A larger open model (gpt-oss-120b on Groq's free tier) answers `data/prompts.txt` in Qud-1's voice. It resumes if it hits a rate limit, so just re-run the cell.

In [ ]:
if os.environ["TEACHER_API_KEY"]:
    !python distill.py

## 2. Fine-tune, publish and convert

Trains the LoRA adapter, merges it, prints sample answers, pushes `quddusr/qud-1`, then builds and pushes `quddusr/qud-1-GGUF` (Q8_0 and Q4_K_M).

In [ ]:
push = "--push quddusr/qud-1 --gguf" if os.environ["HF_TOKEN"] else ""
!python train.py {push}

## 3. Serve it

1. Create a Space at huggingface.co/new-space: name `qud-1`, SDK **Docker**, hardware **CPU basic (free)**.
2. Upload `training/qud-1/space/Dockerfile` and `README.md` to it.
3. In the Space settings, add the secret `LLAMA_API_KEY` (any long random string).
4. In Vercel, add `QUD1_BASE_URL=https://quddusr-qud-1.hf.space/v1` and `QUD1_API_KEY=<the same string>`, then redeploy.

Qud-1's own weights then answer first, with the free larger models as fallback.